<a href="https://colab.research.google.com/github/aniget/SoftUni-AI-Integrations-for-developers/blob/main/Vector%20Databases%2C%20Embeddings%20and%20RAG/RAG_search_books_Anthropic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG on Books with Anthropic Claude

## What this notebook does
We build a **quote-finder** powered by RAG (Retrieval-Augmented Generation):

1. **Index** — Read `.txt` books from Project Gutenberg, split them into overlapping chunks, and store them in ChromaDB as vector embeddings
2. **Search** — Use semantic similarity to find passages matching a theme or topic
3. **RAG loop** — Let Claude autonomously call a `lookup_quote` tool in a multi-turn agentic loop until it has enough context to answer

## Books to download
Download `.txt` files from [Project Gutenberg](https://www.gutenberg.org/browse/scores/top) and upload them to `/content/` in Colab:
- `Metamorphosis.txt`
- `Moby Dick.txt`
- `Alice's Adventures in Wonderland.txt`

In [1]:
# Install required packages:
# - chromadb: vector database for storing and searching book chunks
# - anthropic: the Claude API client
# - rich: pretty terminal tables for displaying search results
!pip install -q chromadb anthropic rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [2]:
import json
from pathlib import Path
from pprint import pprint

# -------------------------------------------------------------------------
# print_response() — displays an Anthropic API response in a readable format
#
# Anthropic's messages.create() returns a Message object with:
#   response.content     → list of content blocks (TextBlock or ToolUseBlock)
#   response.stop_reason → "end_turn" (done) or "tool_use" (wants to call a tool)
#   response.usage       → input_tokens and output_tokens counts
#
# We iterate content blocks and handle each type:
#   TextBlock    → has .text  (the model's reply)
#   ToolUseBlock → has .name and .input (tool call the model wants to make)
# -------------------------------------------------------------------------
def print_response(response):
    print(f"Response ID: {response.id}")
    print(f"Stop Reason: {response.stop_reason}")
    print(f"Input Tokens: {response.usage.input_tokens} | Output Tokens: {response.usage.output_tokens}")
    print()
    print(f"{'=' * 20} [Content Blocks] {'=' * 20}")
    for block in response.content:
        if block.type == "text":
            print(f"[TEXT]\n{block.text}")
        elif block.type == "tool_use":
            print(f"[TOOL CALL] {block.name}")
            print(f"  ID:   {block.id}")
            print(f"  Args: {block.input}")

## Set up ChromaDB

ChromaDB stores text as **vector embeddings** — numerical representations of meaning.
When we search, it compares vectors to find semantically similar passages,
even if the exact words don't match.

`PersistentClient` saves the database to disk so data survives notebook restarts.

In [3]:
from chromadb import PersistentClient

# Path on the Colab VM where ChromaDB stores its files
PATH_TO_CHROMA_DB = "/content/chromadb"
chroma_client = PersistentClient(path=PATH_TO_CHROMA_DB)

# get_or_create_collection is idempotent — safe to run multiple times
chroma_collection = chroma_client.get_or_create_collection(name="books")

## Set up the Anthropic Client

Store your API key in **Colab Secrets** (lock icon in the left sidebar → Add secret named `CLAUDE_API_KEY`).
Never hardcode API keys — they could be leaked if you share the notebook.

In [5]:
from google.colab import userdata
from anthropic import Anthropic

# Retrieve key from Colab Secrets
api_key = userdata.get('CLAUDEAI_API_KEY')
anthropic_client = Anthropic(api_key=api_key)

In [10]:
# Paths to the book files uploaded to Colab
# Upload these via the Files panel (folder icon) on the left sidebar
file_paths = [
    Path("/content/Metamorphosis.txt"),
    Path("/content/Moby Dick.txt"),
    Path("/content/Alice in wonderland.txt")
]

## Index the Books

### What is chunking?
We can't embed an entire book as one vector — it's too long and loses detail.
Instead we split the text into small **chunks** (groups of lines) that each capture a focused idea.

### What is chunk overlap?
Overlapping chunks ensure that context at the boundary between two chunks isn't lost.
For example with `chunk_length=6` and `chunk_overlap=2`:
- Chunk 1: lines 1–6
- Chunk 2: lines 5–10  ← lines 5–6 repeated
- Chunk 3: lines 9–14  ← lines 9–10 repeated

### Chunk ID format
Each chunk gets a unique ID: `BookName_startRow_endRow`
This lets us trace any search result back to its exact location in the book.

In [1]:
from pathlib import Path

def index_file(path_to_file: Path, chunk_length: int = 6, overlap: int = 2):
    """
    Read a text file, split it into overlapping chunks, and store them in ChromaDB.

    Each chunk contains:
        - joined text
        - original line number range for traceability
        - metadata for filtering and display
    """

    # --- Read and clean lines ---
    with path_to_file.open() as f:
        lines = [
            {"text": line.strip(), "row": i + 1}
            for i, line in enumerate(f)
            if line.strip()  # removes blank lines automatically
        ]

    # --- Build chunks ---
    step = chunk_length - overlap
    chunks = []

    for start in range(0, len(lines), step):
        end = min(start + chunk_length, len(lines))
        chunk_lines = lines[start:end]

        chunks.append({
            "text": " ".join(line["text"] for line in chunk_lines),
            "row_range": {
                "from": chunk_lines[0]["row"],
                "to": chunk_lines[-1]["row"]
            }
        })

    # --- Store in ChromaDB ---
    chroma_collection.add(
        ids=[
            f"{path_to_file.stem}_{c['row_range']['from']}_{c['row_range']['to']}"
            for c in chunks
        ],
        metadatas=[
            {
                "book_name": path_to_file.stem,
                "ref_start": c["row_range"]["from"],
                "ref_end": c["row_range"]["to"]
            }
            for c in chunks
        ],
        documents=[c["text"] for c in chunks]
    )

    print(f"Indexed {len(chunks)} chunks from '{path_to_file.name}'")


In [14]:
from rich.console import Console
from rich.table import Table

def print_query_results(result):
    """
    Displays ChromaDB query results as a formatted table using Rich.
    Each row in the table is one matching chunk, identified by its ID.
    """
    console = Console()

    # ChromaDB supports batch queries — result["ids"] is a list of lists
    # One inner list per query sent
    queries_count = len(result["ids"])
    for i in range(queries_count):
        table = Table(show_lines=True, expand=True)
        table.add_column("Chunk ID")   # book + line range
        table.add_column("Text")       # the actual passage

        results_count = len(result["ids"][i])
        for j in range(results_count):
            table.add_row(result["ids"][i][j], result["documents"][i][j])

        console.print(table)

In [16]:
# Index all books — this may take a moment as ChromaDB generates embeddings
# NOTE: If you re-run this cell you'll get a duplicate ID error.
# To reset: delete /content/chromadb and re-run from the PersistentClient cell.
for file_path in file_paths:
    index_file(file_path)

Indexed 519 chunks from 'Metamorphosis.txt'
Indexed 4806 chunks from 'Moby Dick.txt'
Indexed 701 chunks from 'Alice in wonderland.txt'


## Test Semantic Search Directly

Before using Claude, let's verify that ChromaDB's semantic search is working.
We send three queries and inspect which book passages come back.

Notice that the queries don't need to match the exact words in the books —
ChromaDB finds passages with similar *meaning*.

In [17]:
# Test queries — these won't exactly match any book text, but ChromaDB
# will find semantically related passages from whichever book fits best
queries = [
    "The nature is beautiful",
    "In my room is dark",
    "It is amazing how a person can change"
]

# Passing multiple query_texts runs them all in one batch call
semantic_search_results = chroma_collection.query(
    query_texts=queries
)

In [18]:
# Display results as a rich table — one table per query
print_query_results(semantic_search_results)

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Chunk ID              ┃ Text                                                                                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Moby Dick_15173_15179 │ is very sweet and rich; it has been tasted by man; it might do well with strawberries.  │
│                       │ When overflowing with mutual esteem, the whales salute _more hominum_. And thus, though │
│                       │ surrounded by circle upon circle of consternations and affrights, did these inscrutable │
│                       │ creatures at the centre freely and fearlessly indulge in all peaceful concernments;     │
│                       │ yea, serenely revelled                                                                  │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_3482_3487   │ these things unite in a man of greatly superior natural force, with a globular brain    │
│                       │ and a ponderous heart; who has also by the stillness and seclusion of many long         │
│                       │ night-watches in the remotest waters, and beneath constellations never seen here at the │
│                       │ north, been led to think untraditionally and independently; receiving all nature’s      │
│                       │ sweet or savage impressions fresh from her own virgin voluntary and confiding           │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_903_908     │ quietest, most enchanting bit of romantic landscape in all the valley of the Saco. What │
│                       │ is the chief element he employs? There stand his trees, each with a hollow trunk, as if │
│                       │ a hermit and a crucifix were within; and here sleeps his meadow, and there sleep his    │
│                       │ cattle; and up from yonder cottage goes a sleepy smoke. Deep into distant woodlands     │
│                       │ winds a mazy way, reaching to overlapping spurs of mountains bathed in                  │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_17389_17394 │ the trees stood high and haughty, feeling their living sap; the industrious earth       │
│                       │ beneath was as a weaver’s loom, with a gorgeous carpet on it, whereof the ground-vine   │
│                       │ tendrils formed the warp and woof, and the living flowers the figures. All the trees,   │
│                       │ with all their laden branches; all the shrubs, and ferns, and grasses; the              │
│                       │ message-carrying air; all these unceasingly were active. Through the                    │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_18573_18578 │ the world, the Indian ocean and Atlantic being but its arms. The same waves wash the    │
│                       │ moles of the new-built Californian towns, but yesterday planted by the recentest race   │
│                       │ of men, and lave the faded but still gorgeous skirts of Asiatic lands, older than       │
│                       │ Abraham; while all between float milky-ways of coral isles, and low-lying, endless,     │
│                       │ unknown Archipelagoes, and impenetrable Japans. Thus this mysterious, divine            │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_16256_16261 │ a most refreshing, convivial, beautiful object to behold. As its name imports, it is of │
│                       │ an exceedingly rich, mottled t

┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Chunk ID                ┃ Text                                                                                  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Metamorphosis_1284_1289 │ living room being left open every evening. He got into the habit of closely watching  │
│                         │ it for one or two hours before it was opened and then, lying in the darkness of his   │
│                         │ room where he could not be seen from the living room, he could watch the family in    │
│                         │ the light of the dinner table and listen to their conversation—with everyone’s        │
│                         │ permission, in a way, and thus quite differently from before.                         │
├─────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_2748_2753     │ keeping my eyes shut, in order the more to concentrate the snugness of being in bed.  │
│                         │ Because no man can ever feel his own identity aright except his eyes be closed; as if │
│                         │ darkness were indeed the proper element of our essences, though light be more         │
│                         │ congenial to our clayey part. Upon opening my eyes then, and coming out of my own     │
│                         │ pleasant and self-created darkness into the imposed and coarse outer gloom of the     │
├─────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_2752_2757     │ part. Upon opening my eyes then, and coming out of my own pleasant and self-created   │
│                         │ darkness into the imposed and coarse outer gloom of the unilluminated                 │
│                         │ twelve-o’clock-at-night, I experienced a disagreeable revulsion. Nor did I at all     │
│                         │ object to the hint from Queequeg that perhaps it were best to strike a light, seeing  │
│                         │ that we were so wide awake; and besides he felt a strong desire to have a few quiet   │
│                         │ puffs                                                                                 │
├─────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_2638_2643     │ As I sat there in that now lonely room; the fire burning low, in that mild stage      │
│                         │ when, after its first intensity has warmed the air, it then only glows to be looked   │
│                         │ at; the evening shades and phantoms gathering round the casements, and peering in     │
│                         │ upon us silent, solitary twain; the storm booming without in solemn swells; I began   │
│                         │ to be sensible of strange feelings. I felt a melting in me. No more my splintered     │
│                         │ heart                                                                                 │
├─────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_16540_16546   │ seeks the food of light, so he lives in light. He makes his berth an Aladdin’s lamp,  │
│                         │ and lays him down in it; so that in the pitchiest night the ship’s black hull still   │
│                         │ houses an illumination. See with what entire freedom the whaleman takes his handful   │
│                         │ of lamps—often but old bottles and vials, though—to the copper cooler at the          │
│                         │ try-works, and replenishes them there, as mugs of ale at a vat. He                    │
├─────────────────────────┼─────────────────────────────

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Chunk ID                      ┃ Text                                                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Alice in wonderland_1237_1243 │ she had succeeded in bringing herself down to her usual height. It was so long  │
│                               │ since she had been anything near the right size, that it felt quite strange at  │
│                               │ first; but she got used to it in a few minutes, and began talking to herself,   │
│                               │ as usual. “Come, there’s half my plan done now! How puzzling all these changes  │
│                               │ are! I’m never sure what I’m going to be, from one minute to another! However,  │
│                               │ I’ve got back to my                                                             │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ Alice in wonderland_1242_1247 │ done now! How puzzling all these changes are! I’m never sure what I’m going to  │
│                               │ be, from one minute to another! However, I’ve got back to my right size: the    │
│                               │ next thing is, to get into that beautiful garden—how _is_ that to be done, I    │
│                               │ wonder?” As she said this, she came suddenly upon an open place, with a little  │
│                               │ house in it about four feet high. “Whoever lives there,” thought Alice, “it’ll  │
│                               │ never do to come upon them                                                      │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ Alice in wonderland_327_332   │ Alice took up the fan and gloves, and, as the hall was very hot, she kept       │
│                               │ fanning herself all the time she went on talking: “Dear, dear! How queer        │
│                               │ everything is to-day! And yesterday things went on just as usual. I wonder if   │
│                               │ I’ve been changed in the night? Let me think: was I the same when I got up this │
│                               │ morning? I almost think I can remember feeling a little different. But if I’m   │
│                               │ not the same, the next question is, Who                                         │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_3890_3897           │ “Queequeg,” said I, going up to him, “Queequeg, what’s the matter with you?”    │
│                               │ “He hain’t been a sittin’ so all day, has he?” said the landlady. But all we    │
│                               │ said, not a word could we drag out of him; I almost felt like pushing him over, │
│                               │ so as to change his position, for it was almost intolerable, it seemed so       │
│                               │ painfully and unnaturally constrained;                                          │
├───────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_850_855             │ regulating the circulation. Whenever I find myself growing grim about the       │
│                               │ mouth; whenever it is a damp, drizzly November in my soul; whenever I find      │
│                               │ myself involuntarily pausing before coffin warehouses, and bringing up the rear │
│                               │ of every funeral I meet; and especially whenever my hypos get such an upper     │
│                               │ hand of me, that it re

## RAG with Claude — Agentic Tool Loop

Now we wire Claude into the pipeline. Instead of searching manually,
Claude will autonomously decide when and how to call `lookup_quote`.

### The agentic loop
Claude may need to call the tool **multiple times** for complex queries
(e.g. finding quotes on multiple themes and combining them).
We keep calling the API in a loop until Claude's `stop_reason` is `"end_turn"`
rather than `"tool_use"`.

```
User prompt
    ↓
Claude → stop_reason="tool_use" → we run lookup_quote → send result back
    ↓                ↑_______________loop if more tool calls needed__________|
Claude → stop_reason="end_turn" → final answer
```

In [6]:
from pydantic import BaseModel, Field
from typing import List

# -------------------------------------------------------------------------
# QuoteLookupRequest — defines the input schema for our lookup_quote tool
#
# Pydantic's BaseModel gives us:
#   - automatic validation of incoming data
#   - .model_json_schema() to generate the JSON Schema for the Anthropic API
# -------------------------------------------------------------------------


class QuoteLookupRequest(BaseModel):
    """
    Searches the quotes catalog using semantic similarity to find phrases
    matching the user's intent. Use this tool when a user asks about
    the content of a book, a quote reference, or recommendations.
    """
    query: str = Field(
        description="A natural language search query describing the desired quote "
                    "(e.g. 'leadership as the most important characteristic of the modern man', "
                    "'love will save the world', 'motivation is all you need')"
    )

In [20]:
# -------------------------------------------------------------------------
# Build the Anthropic tool schema from the Pydantic model
#
# KEY DIFFERENCE from OpenAI:
#   OpenAI: { "type": "function", "name": ..., "parameters": { ... }, "strict": True }
#   Anthropic: flat dict, no "type" wrapper, uses "input_schema" not "parameters",
#              no "strict" field needed
# -------------------------------------------------------------------------
quote_lookup_schema = QuoteLookupRequest.model_json_schema()

tools = [
    {
        "name": "lookup_quote",
        "description": quote_lookup_schema.get("description", "Search for quotes in the book catalog"),
        "input_schema": {
            "type": "object",
            "properties": quote_lookup_schema["properties"],
            "required": [*quote_lookup_schema["properties"].keys()]
        }
    }
]

In [21]:
def lookup_quote(req: QuoteLookupRequest):
    """
    Executes a semantic search in ChromaDB and returns matching passages.
    The chunk ID encodes the book name and line range, e.g.:
        'Metamorphosis_42_47' → lines 42-47 of Metamorphosis
    """
    query_result = chroma_collection.query(
        query_texts=[req.query],
        include=["documents", "metadatas"]  # include metadatas for book name
    )

    result_count = len(query_result["ids"][0])

    # Return a list of dicts with id, book name, and the passage text
    # The id format is 'BookName_startLine_endLine' — Claude can cite this
    return [
        {
            "id": query_result["ids"][0][i],
            "book": query_result["metadatas"][0][i].get("book_name", "Unknown"),
            "text": query_result["documents"][0][i]
        }
        for i in range(result_count)
    ]


# Registry mapping tool names to their handler functions.
# When Claude calls a tool by name, we look up and invoke the right function.
tool_handlers = {
    "lookup_quote": lambda args: lookup_quote(QuoteLookupRequest(**args))
}

In [22]:
# -------------------------------------------------------------------------
# interact_with_ai() — the agentic loop
#
# This function keeps calling Claude until it produces a final answer.
# It handles the full tool-use cycle:
#
#   1. Send conversation to Claude
#   2. If stop_reason == "tool_use": execute the tool and append the result
#   3. If stop_reason == "end_turn":  return the final conversation
#   4. Repeat up to max_iterations_count times to prevent infinite loops
#
# KEY DIFFERENCES from OpenAI agentic loop:
#
#   OpenAI appends tool results directly:  conversation.append({ "type": "function_call_output", ... })
#   Anthropic wraps them in a user message: { "role": "user", "content": [{ "type": "tool_result", ... }] }
#
#   OpenAI extends conversation with response.output (list)
#   Anthropic appends ONE assistant message: { "role": "assistant", "content": response.content }
#
#   OpenAI detects tool calls via item.type == "function_call"
#   Anthropic detects them via response.stop_reason == "tool_use"
#   (and block.type == "tool_use" per content block)
#
#   OpenAI args: json.loads(tc.arguments)  — raw JSON string needs parsing
#   Anthropic args: block.input            — already a dict, no parsing needed
# -------------------------------------------------------------------------
def interact_with_ai(messages, system, max_iterations_count=10):
    """
    Runs the agentic loop: sends messages to Claude, handles tool calls,
    and continues until Claude produces a final text response.

    Args:
        messages: list of {role, content} dicts (user/assistant turns only)
        system: the system prompt string
        max_iterations_count: safety cap to prevent infinite loops

    Returns:
        The full conversation history including all tool calls and results
    """
    full_conversation = [*messages]  # copy so we don't mutate the original

    for i in range(max_iterations_count):
        print(f"Starting iteration #{i + 1}")

        # Call Claude with the full conversation so far
        current_response = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            system=system,            # system prompt is a separate parameter, not a message
            messages=full_conversation,
            tools=tools,
            max_tokens=1024
        )

        # Append Claude's reply as an assistant turn in the conversation history
        # response.content is a list of content blocks (TextBlock and/or ToolUseBlock)
        full_conversation.append({
            "role": "assistant",
            "content": current_response.content
        })

        # Check if Claude is done — "end_turn" means a final text answer
        if current_response.stop_reason == "end_turn":
            print("Claude finished — end_turn reached.")
            return full_conversation

        # Otherwise Claude wants to call tools — process each ToolUseBlock
        tool_use_blocks = [
            block for block in current_response.content
            if block.type == "tool_use"
        ]

        print(f"Found {len(tool_use_blocks)} tool call(s).")

        # Collect all tool results into a single user message
        # (Anthropic requires all tool_result blocks in ONE user message)
        tool_results = []
        for block in tool_use_blocks:
            print(f"  Calling tool: {block.name} with args: {block.input}")

            handler = tool_handlers[block.name]

            # block.input is already a dict — no json.loads() needed (unlike OpenAI)
            result = handler(block.input)

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,        # links result to the specific tool_use block
                "content": json.dumps(result)   # must be a JSON string
            })

        # Append all tool results as a single user message
        full_conversation.append({
            "role": "user",
            "content": tool_results
        })

    raise Exception(f"The AI interaction couldn't finish in {max_iterations_count} iterations.")

In [23]:
# -------------------------------------------------------------------------
# System prompt and user prompts
#
# KEY DIFFERENCE from OpenAI:
#   OpenAI supports { "role": "developer", "content": ... } inside the messages list
#   Anthropic does NOT — the system prompt goes in the `system=` parameter only
# -------------------------------------------------------------------------
system_prompt = (
    "You are an expert in book recommendations and quote finding. "
    "The user will ask you to lookup quotes on a given topic. "
    "Always use the 'lookup_quote' tool to find interesting references. "
    "The final response should be based solely on the 'lookup_quote' tool call output you get. "
    "Do not invent or refer to any quotes that are not part of the 'lookup_quote' output."
)

# A variety of prompts that test different aspects of the RAG pipeline:
# - simple single-topic search
# - descriptive/atmospheric search
# - multi-topic search requiring multiple tool calls
# - subjective/interpretive search
user_prompts = [
    "I need to find a quote about the transformational power of love and beauty.",
    "Find a picturesque quote about childhood, dreaming and living effortlessly.",
    "Find quotes about the meaning of life and combine them with other quotes about simple existential obstacles to make the reader feel more motivated.",
    "Find the most dark soul breaking quotes in existence."
]

In [24]:
# -------------------------------------------------------------------------
# Run the full RAG pipeline for each user prompt
#
# For each prompt:
#   1. Start a fresh conversation with just the user's question
#   2. Let interact_with_ai() handle the agentic tool loop
#   3. Print the last message in the conversation (Claude's final answer)
#
# The last message is an assistant turn whose content is a list of TextBlocks
# -------------------------------------------------------------------------
for user_prompt in user_prompts:
    print(f"\n{'#' * 60}")
    print(f"USER: {user_prompt}")
    print('#' * 60)

    conversation = interact_with_ai(
        messages=[
            {"role": "user", "content": user_prompt}
            # No "developer" role here — system prompt passed separately
        ],
        system=system_prompt
    )

    # The last message is the final assistant response
    # Its content is a list of blocks — we extract and print all TextBlocks
    final_message = conversation[-1]
    for block in final_message["content"]:
        if hasattr(block, "text"):   # TextBlock has .text; ToolUseBlock does not
            print(block.text)


############################################################
USER: I need to find a quote about the transformational power of love and beauty.
############################################################
Starting iteration #1
Found 1 tool call(s).
  Calling tool: lookup_quote with args: {'query': 'transformational power of love and beauty'}
Starting iteration #2
Claude finished — end_turn reached.
Based on my search, I found several beautiful quotes about beauty, strength, and transformation, particularly from Herman Melville's "Moby Dick." Here are the most relevant ones about the transformational power of love and beauty:

**From Moby Dick:**

*"Real strength never impairs beauty or harmony, but it often bestows it; and in everything imposingly beautiful, strength has much to do with the magic."*

This quote speaks to how true strength and beauty work together transformatively, with strength actually enhancing and creating beauty rather than diminishing it.

*"Loveliness unfathomabl

## Summary: OpenAI vs Anthropic Differences

| Concept | OpenAI | Anthropic |
|---|---|---|
| API method | `responses.create()` | `messages.create()` |
| System prompt | `{"role": "developer", ...}` in messages | `system=` parameter |
| Tool schema key | `"parameters"` | `"input_schema"` |
| Tool type wrapper | `{"type": "function", ...}` | flat dict, no type wrapper |
| `strict` field | supported | not needed |
| Tool args format | JSON string → needs `json.loads()` | already a dict in `block.input` |
| Detect tool call | `item.type == "function_call"` | `response.stop_reason == "tool_use"` |
| Tool result format | `{"type": "function_call_output", "call_id": ...}` | `{"type": "tool_result", "tool_use_id": ...}` |
| Tool result placement | appended directly to messages list | wrapped in a `user` message |
| Append assistant reply | `conversation.extend(response.output)` | `conversation.append({"role": "assistant", "content": response.content})` |
| Final text output | `response.output_text` | iterate `response.content`, print `block.text` |